# 🏠 California Housing 주택 가격 예측 — End-to-End 서비스

California Housing 데이터로 주택 가격을 예측하는 모델을 만들고, 사용자가 웹에서 주택 정보 8개를 입력하면 예측 가격을 돌려주는 서비스를 처음부터 끝까지 만든다.

```
사용자 → Streamlit(:8501) → FastAPI(:8000) → PyTorch 회귀 모델 → 예측 가격 → 화면 표시
```

**실행 방법:** 위에서부터 셀을 순서대로 실행하면 STEP 1~7이 한 번에 끝납니다.

| STEP | 내용 |
|---|---|
| 1 | 프로젝트 구조 준비 + 패키지 설치 |
| 2·3 | 데이터 전처리 + PyTorch 회귀 모델 학습(50 epoch) |
| 4 | 추론 모듈(HousingPredictor) |
| 5 | FastAPI 백엔드(/predict, /health) |
| 6 | Streamlit 프론트엔드(입력폼 8개) |
| 7 | 통합 테스트(정상·sanity·검증·동시요청) |

## STEP 1. 프로젝트 구조 준비

모델·API·UI를 분리해서 관리하도록 폴더를 만들고, 노트북 안에서 FastAPI 서버를 띄우는 도우미(`serve_in_thread`)를 정의한다.

```
app/       housing_model.py · housing_schemas.py · housing_api.py
frontend/  app_housing.py
models/    housing_model.pth · housing_preprocessing.json
```

In [1]:
# 패키지 설치 + 폴더 생성 + 서버 도우미 정의
import subprocess, sys
try:
    import streamlit  # noqa
except ImportError:
    print('streamlit 설치 중... (1분 정도)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit'], check=True)

import os, threading, time, socket, contextlib
import uvicorn

for d in ['app', 'frontend', 'models']:
    os.makedirs(d, exist_ok=True)
open('app/__init__.py', 'a').close()  # app 을 패키지로 인식시키기

_SERVERS = {}

def stop_server(port=8000):
    entry = _SERVERS.pop(port, None)
    if entry:
        server, thread = entry
        server.should_exit = True
        for _ in range(50):
            if not thread.is_alive():
                break
            time.sleep(0.1)

def serve_in_thread(app_path, host='127.0.0.1', port=8000, log_level='warning'):
    """노트북 안에서 uvicorn 서버를 백그라운드 스레드로 띄운다."""
    stop_server(port)
    config = uvicorn.Config(app_path, host=host, port=port, log_level=log_level)
    server = uvicorn.Server(config)
    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()
    for _ in range(100):
        with contextlib.closing(socket.socket()) as s:
            if s.connect_ex((host, port)) == 0:
                break
        time.sleep(0.1)
    _SERVERS[port] = (server, thread)
    print(f'✅ 서버 실행: http://{host}:{port}')

print('✅ STEP 1 완료 — 폴더·서버 도우미 준비됨')

streamlit 설치 중... (1분 정도)
✅ STEP 1 완료 — 폴더·서버 도우미 준비됨


## STEP 2·3. 데이터 전처리 + 회귀 모델 학습

먼저 모델 클래스(`HousingRegressionModel`)와 추론 래퍼(`HousingPredictor`)를 `app/housing_model.py`에 저장한다. 학습과 추론이 **같은 모델 정의**를 쓰도록 하기 위해서다.

> ⚠️ **가장 중요:** 학습 때 쓴 `mean`/`std`와 **Feature 순서**를 반드시 저장해서, 추론 때도 똑같이 사용한다. API에서 새로 계산하면 코드는 돌아가도 예측이 틀리는 **Silent Error**가 난다.

In [2]:
%%writefile app/housing_model.py
import json
import numpy as np
import torch
import torch.nn as nn

FEATURE_NAMES = ["MedInc", "HouseAge", "AveRooms", "AveBedrms",
                 "Population", "AveOccup", "Latitude", "Longitude"]

class HousingRegressionModel(nn.Module):
    def __init__(self, in_features=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x)

class HousingPredictor:
    def __init__(self, model_path="models/housing_model.pth",
                 prep_path="models/housing_preprocessing.json"):
        with open(prep_path, encoding="utf-8") as f:
            prep = json.load(f)
        self.feature_names = prep["feature_names"]
        self.mean = np.array(prep["mean"], dtype=np.float32)
        self.std = np.array(prep["std"], dtype=np.float32)
        self.model = HousingRegressionModel(len(self.feature_names))
        self.model.load_state_dict(torch.load(model_path, map_location="cpu"))
        self.model.eval()
    def predict(self, features: dict) -> float:
        x = np.array([[float(features[n]) for n in self.feature_names]], dtype=np.float32)
        x = (x - self.mean) / self.std
        with torch.no_grad():
            return self.model(torch.from_numpy(x)).item()


Writing app/housing_model.py


In [3]:
# 데이터 로딩 → 정규화 → 50 epoch 학습 → 모델·전처리값 저장
import json, numpy as np, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from app.housing_model import HousingRegressionModel, FEATURE_NAMES

torch.manual_seed(42); np.random.seed(42)

data = fetch_california_housing()
print('Feature 순서:', list(data.feature_names))
assert list(data.feature_names) == FEATURE_NAMES, 'Feature 순서 불일치!'

X = data.data.astype('float32')   # (20640, 8)
y = data.target.astype('float32') # 중위 주택 가격 (단위 $100,000)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 학습 데이터 기준 mean/std (반드시 저장해서 추론 때 재사용)
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train_n = (X_train - mean) / std
X_test_n = (X_test - mean) / std

train_ds = TensorDataset(torch.tensor(X_train_n), torch.tensor(y_train).view(-1, 1))
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)

model = HousingRegressionModel(8)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(50):
    model.train()
    for xb, yb in train_dl:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_mse = criterion(model(torch.tensor(X_test_n)), torch.tensor(y_test).view(-1, 1)).item()
        print(f'epoch {epoch+1:2d}  test MSE {test_mse:.4f}')

torch.save(model.state_dict(), 'models/housing_model.pth')
json.dump({'feature_names': FEATURE_NAMES, 'mean': mean.tolist(), 'std': std.tolist()},
          open('models/housing_preprocessing.json', 'w'))
print('✅ 모델 + 전처리값 저장 완료')

Feature 순서: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
epoch 10  test MSE 0.3211
epoch 20  test MSE 0.2987
epoch 30  test MSE 0.2955
epoch 40  test MSE 0.2963
epoch 50  test MSE 0.2811
✅ 모델 + 전처리값 저장 완료


## STEP 4. 추론 모듈 확인

`HousingPredictor`는 저장된 모델·전처리값을 불러와, 입력 dict를 **Feature 순서대로 정렬 → 정규화 → 예측**한다. 아래에서 단독으로 잘 동작하는지, 그리고 소득이 오르면 가격이 오르는지(Sanity)를 확인한다.

In [4]:
from app.housing_model import HousingPredictor

predictor = HousingPredictor()
base = dict(MedInc=4.0, HouseAge=25, AveRooms=5.5, AveBedrms=1.1,
            Population=1200, AveOccup=3.0, Latitude=34.0, Longitude=-118.0)

val = predictor.predict(base)
print(f'기본 입력 예측: {val:.3f}  (≈ ${val*100000:,.0f})')

hi = predictor.predict(dict(base, MedInc=8.0))
lo = predictor.predict(dict(base, MedInc=1.0))
print(f'Sanity: MedInc 8.0 → {hi:.3f}  vs  MedInc 1.0 → {lo:.3f}  ('
      + ('PASS ✅' if hi > lo else 'FAIL ❌') + ')')

기본 입력 예측: 2.342  (≈ $234,177)
Sanity: MedInc 8.0 → 3.583  vs  MedInc 1.0 → 1.200  (PASS ✅)


## STEP 5. FastAPI 백엔드

입력 검증(Pydantic Schema)과 예측 엔드포인트(`POST /predict`), 상태 확인(`GET /health`)을 만든다. Latitude(32~42)·Longitude(-125~-114)·양수 제약을 스키마에서 검증하므로, 잘못된 값은 모델에 닿기 전에 **422**로 막힌다.

In [5]:
%%writefile app/housing_schemas.py
from pydantic import BaseModel, Field

class HousingInput(BaseModel):
    MedInc: float = Field(..., gt=0, description="중위 소득 (양수)")
    HouseAge: float = Field(..., ge=0, le=100, description="주택 평균 연식")
    AveRooms: float = Field(..., gt=0, description="평균 방 개수")
    AveBedrms: float = Field(..., gt=0, description="평균 침실 개수")
    Population: float = Field(..., gt=0, description="지역 인구")
    AveOccup: float = Field(..., gt=0, description="평균 거주 인원")
    Latitude: float = Field(..., ge=32, le=42, description="위도 (California 범위)")
    Longitude: float = Field(..., ge=-125, le=-114, description="경도 (California 범위)")

class HousingPrediction(BaseModel):
    predicted_value: float
    predicted_price_usd: float


Writing app/housing_schemas.py


In [6]:
%%writefile app/housing_api.py
import logging, time
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from app.housing_schemas import HousingInput, HousingPrediction
from app.housing_model import HousingPredictor

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("housing_api")

app = FastAPI(title="California Housing Price API", version="1.0.0")

@app.middleware("http")
async def log_requests(request: Request, call_next):
    start = time.time()
    resp = await call_next(request)
    logger.info(f"{request.method} {request.url.path} -> {resp.status_code} ({(time.time()-start)*1000:.1f}ms)")
    return resp

@app.exception_handler(Exception)
async def global_handler(request: Request, exc: Exception):
    logger.error(f"에러: {exc}")
    return JSONResponse(status_code=500, content={"detail": "서버 내부 오류가 발생했습니다."})

predictor = None

@app.on_event("startup")
def load_model():
    global predictor
    predictor = HousingPredictor()
    logger.info("HousingPredictor 로드 완료")

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": predictor is not None}

@app.post("/predict", response_model=HousingPrediction)
def predict(inp: HousingInput):
    value = predictor.predict(inp.model_dump())
    return HousingPrediction(predicted_value=value, predicted_price_usd=value * 100000)


Writing app/housing_api.py


In [7]:
# 백엔드를 8000 포트에 띄우고 확인
import requests, time
serve_in_thread('app.housing_api:app', port=8000)
time.sleep(2)
print('health:', requests.get('http://localhost:8000/health').json())
sample = dict(MedInc=4.0, HouseAge=25, AveRooms=5.5, AveBedrms=1.1,
              Population=1200, AveOccup=3.0, Latitude=34.0, Longitude=-118.0)
print('predict:', requests.post('http://localhost:8000/predict', json=sample).json())

✅ 서버 실행: http://127.0.0.1:8000
health: {'status': 'ok', 'model_loaded': True}
predict: {'predicted_value': 2.3417718410491943, 'predicted_price_usd': 234177.18410491943}


## STEP 6. Streamlit 프론트엔드

주택 정보 8개 입력폼을 만들고, `가격 예측` 버튼을 누르면 FastAPI `/predict`를 호출해 결과를 보여준다.

> ⚠️ 입력 dict의 **Key 이름은 FastAPI Schema와 완전히 같아야** 한다(데이터 계약). `MedInc`를 `med_income`으로 보내면 안 된다.

In [8]:
%%writefile frontend/app_housing.py
"""
DP - California Housing 가격 예측 대시보드 (Streamlit)
FastAPI 백엔드(/predict)와 연동한다.
"""
import streamlit as st
import requests

st.set_page_config(page_title="California Housing 가격 예측", page_icon="🏠", layout="centered")
API = "http://localhost:8000"

st.title("🏠 California Housing 주택 가격 예측")
st.caption("주택 정보 8개를 입력하면 예측 가격을 반환합니다.")

# 서버 상태
try:
    h = requests.get(f"{API}/health", timeout=5).json()
    st.success("🟢 서버 연결됨") if h.get("status") == "ok" else st.warning("🟡 모델 로딩 중")
except Exception:
    st.error("🔴 서버 연결 실패 — 백엔드 셀을 먼저 실행하세요.")

# 8개 입력 (Key 이름은 FastAPI Schema와 완전히 동일해야 함 = 데이터 계약)
defaults = {
    "MedInc": 4.0, "HouseAge": 25.0, "AveRooms": 5.5, "AveBedrms": 1.1,
    "Population": 1200.0, "AveOccup": 3.0, "Latitude": 34.0, "Longitude": -118.0,
}
help_txt = {
    "MedInc": "지역 중위 소득", "HouseAge": "주택 평균 연식", "AveRooms": "평균 방 개수",
    "AveBedrms": "평균 침실 개수", "Population": "지역 인구", "AveOccup": "평균 거주 인원",
    "Latitude": "위도 (32~42)", "Longitude": "경도 (-125~-114)",
}
c1, c2 = st.columns(2)
values = {}
for i, key in enumerate(defaults):
    col = c1 if i % 2 == 0 else c2
    values[key] = col.number_input(key, value=defaults[key], help=help_txt[key], format="%.4f")

if st.button("💰 가격 예측", type="primary", use_container_width=True):
    try:
        resp = requests.post(f"{API}/predict", json=values, timeout=30)
        if resp.status_code == 200:
            st.session_state["last"] = resp.json()
        else:
            detail = resp.json().get("detail", "")
            st.error(f"❌ 입력 오류 (HTTP {resp.status_code})")
            st.caption(str(detail))
    except requests.exceptions.ConnectionError:
        st.error("🔌 서버에 연결할 수 없습니다.")
    except Exception as e:
        st.error(f"❌ 오류: {type(e).__name__}")

if "last" in st.session_state:
    d = st.session_state["last"]
    st.divider()
    st.metric("예상 주택 가격", f"${d['predicted_price_usd']:,.0f}")
    st.caption(f"모델 출력값 {d['predicted_value']:.3f} × $100,000")


Writing frontend/app_housing.py


In [9]:
# 대시보드(Streamlit) 8501 포트에 띄우고 노트북 안에 iframe으로 표시
import subprocess, time
subprocess.Popen(['streamlit', 'run', 'frontend/app_housing.py',
                  '--server.port', '8501', '--server.headless', 'true',
                  '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false'])
time.sleep(12)
from google.colab.output import serve_kernel_port_as_iframe
print('👇 아래 대시보드에서 값을 넣고 [💰 가격 예측]을 눌러보세요 (안 뜨면 이 셀만 다시 실행)')
serve_kernel_port_as_iframe(8501, height=760)

👇 아래 대시보드에서 값을 넣고 [💰 가격 예측]을 눌러보세요 (안 뜨면 이 셀만 다시 실행)


<IPython.core.display.Javascript object>

## STEP 7. 통합 테스트

모델·API·검증이 하나의 시스템으로 정상 동작하는지 확인한다. 정상 예측, Sanity(소득↑→가격↑), 필드 누락·범위 위반·음수값 방어(4xx), 동시 요청 8개, Health Check까지 자동으로 검사한다.

In [10]:
import requests, concurrent.futures, time
BASE = 'http://localhost:8000'
valid = dict(MedInc=4.0, HouseAge=25, AveRooms=5.5, AveBedrms=1.1,
             Population=1200, AveOccup=3.0, Latitude=34.0, Longitude=-118.0)

# Test 1. 정상 예측
r = requests.post(f'{BASE}/predict', json=valid)
print('Test1 정상  :', r.status_code, r.json())

# Test 2. Sanity (소득↑ → 가격↑)
hi = requests.post(f'{BASE}/predict', json=dict(valid, MedInc=8.0)).json()['predicted_value']
lo = requests.post(f'{BASE}/predict', json=dict(valid, MedInc=1.0)).json()['predicted_value']
print(f'Test2 sanity: 8.0→{hi:.3f} > 1.0→{lo:.3f}  ' + ('PASS ✅' if hi > lo else 'FAIL ❌'))

# Test 3. 필드 누락 → 422
miss = dict(valid); del miss['MedInc']
print('Test3 필드누락:', requests.post(f'{BASE}/predict', json=miss).status_code, '(422 기대)')

# Test 4. Latitude 범위 이탈(50) → 422
print('Test4 Lat50 :', requests.post(f'{BASE}/predict', json=dict(valid, Latitude=50)).status_code, '(422 기대)')

# Test 5. 음수 소득 → 422
print('Test5 음수소득:', requests.post(f'{BASE}/predict', json=dict(valid, MedInc=-1)).status_code, '(422 기대)')

# Test 6. 동시 요청 8개
def call(_):
    return requests.post(f'{BASE}/predict', json=valid).status_code
t = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as ex:
    results = list(ex.map(call, range(8)))
print(f'Test6 동시8개: {results}  ({time.time()-t:.2f}s)')

# Test 7. Health Check
print('Test7 health:', requests.get(f'{BASE}/health').json())

Test1 정상  : 200 {'predicted_value': 2.3417718410491943, 'predicted_price_usd': 234177.18410491943}
Test2 sanity: 8.0→3.583 > 1.0→1.200  PASS ✅
Test3 필드누락: 422 (422 기대)
Test4 Lat50 : 422 (422 기대)
Test5 음수소득: 422 (422 기대)
Test6 동시8개: [200, 200, 200, 200, 200, 200, 200, 200]  (0.03s)
Test7 health: {'status': 'ok', 'model_loaded': True}


## ✅ 최종 산출물 & 회고

**만들어진 파일**
```
app/housing_model.py       모델 정의 + HousingPredictor
app/housing_schemas.py     Pydantic 입력/출력 검증
app/housing_api.py         FastAPI (/predict, /health)
frontend/app_housing.py    Streamlit 입력폼
models/housing_model.pth           학습된 가중치
models/housing_preprocessing.json  mean·std·feature 순서
```

**이 과제의 핵심 4가지 (정확도보다 중요)**

1. **학습·추론 전처리 일치** — 학습 때 `mean`/`std`를 저장해 추론에서 그대로 재사용했다. API에서 새로 계산하면 Silent Error가 난다.
2. **API 입력 방어** — Latitude 32~42, Longitude -125~-114, 양수 제약을 Pydantic으로 검증해 잘못된 값을 422로 막았다.
3. **데이터 계약 일치** — 프론트엔드가 보내는 Key가 FastAPI Schema 이름과 완전히 같아, Feature 이름·순서가 어긋나지 않는다.
4. **정상 + 에러 + 동시요청 테스트** — 필드 누락·범위 위반·음수값을 막고, 에러 후에도 서버가 살아있으며, 동시 8개 요청을 처리하는 것까지 확인했다.

> 배운 것: "노트북 안에서만 돌던 모델"을 전처리 저장 → 추론 모듈 → API → 프론트엔드로 감싸 실제 서비스 형태로 꺼내는 전체 흐름.